# Neural experiments on Colab (DistilBERT, BiLSTM)

Runs the GPU half of the paper experiments. **Runtime → Change runtime type → T4 GPU.**

Results and the corpus cache go to Google Drive, so if the session disconnects, re-run all cells: finished runs are skipped.
When done, copy `results/` from Drive into your local clone and run `python scripts/analyze.py`.

Approximate T4 times: DistilBERT ~15 min per training run, BiLSTM ~3 min.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/fake-news-paper'
os.environ['FND_RESULTS_DIR'] = f'{WORK}/results'
os.environ['FND_CACHE_DIR'] = f'{WORK}/cache'
os.makedirs(WORK, exist_ok=True)

In [ ]:
!git clone -q -b paper https://github.com/nabilah-hossain-ruba/fake-news-detection.git /content/repo || git -C /content/repo pull -q
%cd /content/repo
!pip install -q datasketch captum tabulate kagglehub

## 1. Build the corpus (first run only, ~10 min; cached on Drive)

In [ ]:
!python -c "import sys; sys.path.insert(0, 'src'); from fnd.data import load_corpus; print(load_corpus()[1])"

## 2. In-domain: DistilBERT (3 seeds) and BiLSTM (5 seeds)

Seed 0 also runs Integrated Gradients on 300 test articles.

In [ ]:
!python scripts/run_neural.py --model distilbert --setting in-domain --seeds 0 --explain 300
!python scripts/run_neural.py --model distilbert --setting in-domain --seeds 1 2

In [ ]:
!python scripts/run_neural.py --model bilstm --setting in-domain --seeds 0 1 2 3 4

## 3. Cross-source: train on two sources, test on the third

In [ ]:
!python scripts/run_neural.py --model distilbert --setting cross-source --seeds 0
!python scripts/run_neural.py --model bilstm --setting cross-source --seeds 0

## 4. Optional: more DistilBERT seeds if you have GPU time left

In [ ]:
!python scripts/run_neural.py --model distilbert --setting in-domain --seeds 3 4